In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch import Tensor
from matplotlib import pyplot as plt
from sklearn.datasets import make_moons, make_circles

In [2]:
class Flow(nn.Module): 
    '''
    Defines the architecture for flow matching
    '''

    def __init__(self, dim: int = 2, h: int = 64, cond: bool = False): # fully connected neural network for learning velocity field 
        super().__init__()
        
        self.cond = cond 
        if self.cond: 
            print("Conditioning model on xc")
            n = 2
        else: 
            print("Neglect condition")
            n = 1
            
        self.net = nn.Sequential(
            nn.Linear(n * dim + 1, h),  
            nn.ELU(),              
            nn.Linear(h, h),       
            nn.ELU(),             
            nn.Linear(h, h),      
            nn.ELU(),              
            nn.Linear(h, dim)       
        )
    
    def forward(self, t: Tensor, xt: Tensor, xc: Tensor) -> Tensor:
        '''
        Concatenates time and position vectors for forward pass through neural network  
        '''

        if self.cond: 
            combined_input = torch.cat((t, xc, xt), -1)
        else:
            combined_input = torch.cat((t, xt), -1)
            
        return self.net(combined_input)
    
    def step(self, xt: Tensor, xc: Tensor, t_start: Tensor, t_end: Tensor) -> Tensor: 
        '''
        Performs midpoint integration to sample transform x0 to x1 (i.e., sample from p1, starting from p0)
        '''
      
        t_start = t_start.view(1, 1).expand(xt.shape[0], 1)
        midpoint_time = t_start + (t_end - t_start) / 2 # evalute the midpoint 
        if self.cond:
            intermediate_value = self(xt=xt, t=t_start, xc=xc) # compute the velocity at this point using the trained neural network 
        else: 
            intermediate_value = self(xt=xt, t=t_start, xc=xc) # compute the velocity at this point using the trained neural network
        updated_xt = xt + intermediate_value * (t_end - t_start) / 2 # move xt using computed velocity 
        midpoint_output = self(t=midpoint_time, xt=updated_xt, xc=xc) # compute vt again 
        
        return xt + (t_end - t_start) * midpoint_output # compute xt

In [3]:
# Training loop
def train():
    '''
    Defines the training procedure for the neural network
    '''
    for _ in range(1000000): #10000000
        
    
        x1 = Tensor(make_moons(1, noise=0.05)[0]) #Tensor([1, 2, 3, 4, 5, 6]).unsqueeze(0) #
        x0 = torch.randn_like(x1)
        noise_scale = 0.002
        xc = x1 + noise_scale * torch.randn_like(x1) #Tensor([1, 0, 3, 0, 5, 0]).unsqueeze(0)
        t = torch.rand(len(x1), 1) #torch.rand(1).unsqueeze(0) #

        xt = (1 - t) * x0 + t * x1

        # Compute loss 
        loss = loss_fn(flow(t=t, xt=xt, xc=xc), x1 - x0)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        print(f"Loss at epoch {_}: {loss.item()}")

        if loss.item() < 0.000001:
            break 
        
   

# Initialize model and optimizer
flow = Flow()
optimizer = torch.optim.Adam(flow.parameters(), 1e-3)
loss_fn = nn.MSELoss()

for _ in range(10):
    train()


Neglect condition
Loss at epoch 0: 0.9289849400520325
Loss at epoch 1: 1.3285374641418457
Loss at epoch 2: 1.0646169185638428
Loss at epoch 3: 0.28050661087036133
Loss at epoch 4: 1.309699535369873
Loss at epoch 5: 0.21366532146930695
Loss at epoch 6: 1.5849848985671997
Loss at epoch 7: 2.368528127670288
Loss at epoch 8: 0.5920508503913879
Loss at epoch 9: 1.50330650806427
Loss at epoch 10: 0.4936833381652832
Loss at epoch 11: 0.41179272532463074
Loss at epoch 12: 1.7447216510772705
Loss at epoch 13: 0.004559107590466738
Loss at epoch 14: 0.4254488945007324
Loss at epoch 15: 1.0419654846191406
Loss at epoch 16: 0.12107384204864502
Loss at epoch 17: 0.039797551929950714
Loss at epoch 18: 0.5204538106918335
Loss at epoch 19: 0.5877475738525391
Loss at epoch 20: 1.792141318321228
Loss at epoch 21: 0.3547933101654053
Loss at epoch 22: 1.405618667602539
Loss at epoch 23: 1.4762284755706787
Loss at epoch 24: 0.4980325996875763
Loss at epoch 25: 0.13504251837730408
Loss at epoch 26: 0.4582346

In [4]:
flow.eval()
x = []
for n in range(1000):
    x1 = torch.Tensor([[0.0883, 0.4923]])
    x0 = x0 = torch.randn_like(x1)
    g = flow.step(xt=x0, xc=x1, t_start=torch.Tensor([0]), t_end=torch.Tensor([1]))
    g = g.detach().numpy().squeeze()
    x.append(g)
print(x)
plt.plot(n,x[0,0])

[array([0.1195856, 0.5034305], dtype=float32), array([-0.03861231,  0.5152728 ], dtype=float32), array([0.02380729, 0.582029  ], dtype=float32), array([-0.0364877,  0.5172537], dtype=float32), array([0.1232276 , 0.51822484], dtype=float32), array([0.0478622 , 0.52697474], dtype=float32), array([-0.04034698,  0.6142122 ], dtype=float32), array([-0.04941371,  0.63592637], dtype=float32), array([-0.0065546 ,  0.54616624], dtype=float32), array([0.01151717, 0.55870587], dtype=float32), array([0.05852306, 0.5148348 ], dtype=float32), array([0.05146611, 0.48776808], dtype=float32), array([-0.05574641,  0.6029122 ], dtype=float32), array([-0.02382976,  0.5523478 ], dtype=float32), array([-0.01181352,  0.64421743], dtype=float32), array([0.08067414, 0.5160421 ], dtype=float32), array([0.09895107, 0.504838  ], dtype=float32), array([0.0909332 , 0.59251374], dtype=float32), array([0.04757491, 0.48886773], dtype=float32), array([-0.02102068,  0.70581627], dtype=float32), array([-0.02800155,  0.52

TypeError: list indices must be integers or slices, not tuple